# Importing Libraries

In [ ]:
import torch
import torchvision
from torchvision import datasets
from torchvision.datasets import MNIST
import matplotlib.pyplot as plt
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import random_split
from torch.utils.data import DataLoader, ConcatDataset, Subset
import torch.nn.functional as F
import os
import wandb
import random
import numpy as np
from sklearn.model_selection import KFold
from torchvision import models
import torch.optim as optim

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset, ConcatDataset
from torchvision import datasets, transforms
from sklearn.model_selection import KFold
import wandb
import numpy as np
from google.colab import userdata
import os
import random
from tqdm.auto import tqdm

# WandB Config

In [ ]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# os.environ['WANDB_API_KEY'] = user_secrets.get_secret("WANDB_API_KEY")
# wandb.login()

In [ ]:
# 1. Configuration
CONFIG = dict({
    "PROJECT_NAME": "resnet50-castorv2",
    "K_FOLDS": 2,
    "NUM_EPOCHS": 2,
    "BATCH_SIZE": 16,
    "LEARNING_RATE": 0.001,
    "INPUT_SIZE": 224*224,
    "NUM_CLASSES": 3,
    "IMAGE_SIZE": 224,
    "ACTIVE": False,
})

# Setting Random Seed

In [ ]:
RANDOM_STATE = 42

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
random.seed(hash("setting random seeds") % 2**32 - 1)
np.random.seed(hash("improves reproducibility") % 2**32 - 1)
torch.manual_seed(hash("by removing stochasticity") % 2**32 - 1)
torch.cuda.manual_seed_all(hash("so runs are repeatable") % 2**32 - 1)

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Loading the Dataset

## Image Transformation

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),          # resize image
    transforms.ToTensor(),                  # convert to tensor [0,1]
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],         # ImageNet mean
        std=[0.229, 0.224, 0.225]           # ImageNet std
    )
])

In [ ]:
dataset_path = "/kaggle/input/castor-v2-224x224/castor_v2_224x224"
# dataset_path = "/kaggle/input/castor-v2-segmented/castor_v2_segmented"
castorv2_dataset = datasets.ImageFolder(dataset_path, transform = transform)

len(castorv2_dataset)

In [ ]:
split_size = 0.8; total_images_len = len(castorv2_dataset)
train_size = int(split_size*total_images_len); val_size = total_images_len - train_size
train_data, validation_data = random_split(castorv2_dataset, [train_size, val_size])
print("Train Images:", len(train_data))
print("Validation Images:", len(validation_data))

In [ ]:
batch_size = 16
train_loader = DataLoader(train_data, batch_size=batch_size,
                          shuffle=True, num_workers=4,
                          pin_memory=True)

val_loader = DataLoader(validation_data, batch_size=batch_size,
                        num_workers=4, pin_memory=True)

## Get Dataset method

In [ ]:
def get_dataset(
    # dataset_path = "/kaggle/input/castor-v2-224x224/castor_v2_224x224",
    dataset_path = "/kaggle/input/castor-v2-segmented/castor_v2_segmented",
    split_size = 0.8
    ):
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),          # resize image
        transforms.ToTensor(),                  # convert to tensor [0,1]
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],         # ImageNet mean
            std=[0.229, 0.224, 0.225]           # ImageNet std
        )
    ])

    castorv2_dataset = datasets.ImageFolder(dataset_path, transform = transform)

    split_size = split_size; total_images_len = len(castorv2_dataset)
    train_size = int(split_size*total_images_len); val_size = total_images_len - train_size
    train_data, validation_data = random_split(castorv2_dataset, [train_size, val_size])

    full_dataset = ConcatDataset([train_data, validation_data])
    
    return full_dataset

# Model

In [ ]:
num_classes = 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, num_classes)  # change head FIRST
model = model.to(device)  # THEN move to GPU

# Fit method

In [ ]:
import torch
import torch.nn as nn
from tqdm import tqdm

def fit(epochs, lr, model, train_loader, val_loader, opt_func=torch.optim.SGD):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.CrossEntropyLoss()

    if opt_func is torch.optim.SGD:
        optimizer = opt_func(model.parameters(), lr=lr, momentum=0.9)
    else:
        optimizer = opt_func(model.parameters(), lr=lr)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        running_corrects = 0
        total = 0

        # --------------------------
        # TRAINING WITH TQDM
        # --------------------------
        train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} Training")
        for images, labels in train_loop:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            _, preds = torch.max(outputs, 1)
            running_corrects += (preds == labels).sum().item()
            total += labels.size(0)

            train_loop.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{running_corrects/total:.4f}"
            })

        train_loss = running_loss / total
        train_acc = running_corrects / total

        # --------------------------
        # VALIDATION WITH TQDM
        # --------------------------
        model.eval()
        val_running_loss = 0.0
        val_running_corrects = 0
        val_total = 0

        val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} Validating")
        with torch.no_grad():
            for images, labels in val_loop:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_running_loss += loss.item() * labels.size(0)
                _, preds = torch.max(outputs, 1)
                val_running_corrects += (preds == labels).sum().item()
                val_total += labels.size(0)

                val_loop.set_postfix({
                    "loss": f"{loss.item():.4f}",
                    "acc": f"{val_running_corrects/val_total:.4f}"
                })

        val_loss = val_running_loss / val_total
        val_acc = val_running_corrects / val_total

        # print(f"\nEpoch {epoch+1}/{epochs} Summary:")
        # print(f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}")
        # print(f"Val   Loss: {val_loss:.4f}  Val   Acc: {val_acc:.4f}\n")

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

    return history


# Training (Skip to KFold)

In [ ]:
history = fit(epochs=40, lr=1e-3, model=model,
              train_loader=train_loader,
              val_loader=val_loader,
              opt_func=torch.optim.SGD)

In [ ]:
# Plot training history: validation loss and accuracy
val_losses = history['val_loss']
val_accs = history['val_acc']
train_losses = history['train_loss']
train_accs = history['train_acc']
epochs = range(1, len(history['val_loss']) + 1)

plt.figure(figsize=(12, 4))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, marker='o', color="orange", label='Train Loss')
plt.plot(epochs, val_losses, marker='o', color="blue" ,label='Val Loss')
plt.title('Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()

# Accuracy plot
plt.subplot(1, 2, 2)
plt.plot(epochs, train_accs, marker='o', color='orange', label='Train Accuracy')
plt.plot(epochs, val_accs, marker='o', color='blue', label='Val Accuracy')
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

# K-Fold

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score
)
import matplotlib.pyplot as plt
import wandb
import copy
# from thop import profile


def plot_confusion_matrix(y_true, y_pred, class_names, normalize=False):
    from sklearn.metrics import confusion_matrix

    labels = np.arange(len(class_names))
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
        cm_display = np.round(cm * 100, 1)
        fmt = ".1f"
    else:
        cm_display = cm
        fmt = "d"

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")

    cbar = ax.figure.colorbar(im, ax=ax)
    cbar.ax.set_ylabel("Count" if not normalize else "Percentage", rotation=-90, va="bottom")

    ax.set(
        xticks=np.arange(len(class_names)),
        yticks=np.arange(len(class_names)),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel="Actual",
        xlabel="Predicted",
        title="Confusion Matrix",
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i,
                format(cm_display[i, j], fmt),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=10,
            )

    fig.tight_layout()
    return fig


def compute_gflops(model, device, input_size):
    model_eval_state = model.training
    model.eval()
    dummy_input = torch.randn(1, 3, input_size, input_size).to(device)
    flops, params = profile(model, inputs=(dummy_input,), verbose=False)
    if model_eval_state:
        model.train()
    gflops = flops / 1e9
    return gflops

def train_fold(fold_idx, train_loader, val_loader, group_name):
    """ Trains a single fold and logs it as a run within the group. """

    
    if CONFIG['ACTIVE']:
        wandb.init(
            project=CONFIG["PROJECT_NAME"],
            group=group_name,
            name=f"fold-{fold_idx}",
            job_type="train",
            config=CONFIG,
            reinit=True
        )

    # Initialize model & move to GPU
    num_classes = 3
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=CONFIG["LEARNING_RATE"])

    # CALLBACKS
    best_val_loss = np.inf
    best_weights = None           # keep in-memory only during training
    patience = 8
    patience_counter = 0
    best_epoch = -1  # <-- track which epoch gave best model

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        patience=3,
        factor=0.3,
        verbose=True
    )

    for epoch in range(CONFIG["NUM_EPOCHS"]):
        # ------------------- TRAIN -------------------
        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0

        progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['NUM_EPOCHS']}")

        for batch_X, batch_y in progress:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()

            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted_train = torch.max(outputs.data, 1)
            total_train += batch_y.size(0)
            correct_train += (predicted_train == batch_y).sum().item()

        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = correct_train / total_train

        # ------------------- VALIDATE -------------------
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        all_labels = []
        all_preds = []
        all_probs = []

        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)

                loss = criterion(outputs, batch_y)
                val_loss += loss.item()

                probs = torch.softmax(outputs, dim=1)
                _, predicted = torch.max(outputs.data, 1)

                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()

                all_labels.append(batch_y.cpu().numpy())
                all_preds.append(predicted.cpu().numpy())
                all_probs.append(probs.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = correct / total

        all_labels = np.concatenate(all_labels)
        all_preds = np.concatenate(all_preds)
        all_probs = np.concatenate(all_probs)

        precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
        recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)

        try:
            roc_auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
        except:
            roc_auc = float("nan")

        cm_fig = plot_confusion_matrix(
            all_labels,
            all_preds,
            ["Healthy Leafs", "Semilooper", "Spodoptera"],
            normalize=False
        )

        # ------------------- CALLBACKS  -------------------

        # SAVE BEST MODEL (in-memory only — do NOT save or log artifact here)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_epoch = epoch + 1  # 1-based
            best_weights = copy.deepcopy(model.state_dict())   # keep best weights in RAM
            print(f"   ✔ New best model found at epoch {best_epoch} (val_loss={best_val_loss:.4f})")
        else:
            patience_counter += 1
            print(f"   patience: {patience_counter}/{patience}")

        # EARLY STOPPING
        if patience_counter >= patience:
            print("⛔ Early stopping triggered — patience exceeded")
            break

        # REDUCE LR ON PLATEAU
        scheduler.step(avg_val_loss)

        if CONFIG['ACTIVE']:
            # ------------------- W&B LOG -------------------
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": avg_train_loss,
                "train_accuracy": train_accuracy,
                "val_loss": avg_val_loss,
                "val_accuracy": val_accuracy,
                "val_precision": precision,
                "val_recall": recall,
                "val_roc_auc": roc_auc,
                "current_lr": optimizer.param_groups[0]["lr"],
                "confusion_matrix": cm_fig,
                "fold": fold_idx
            })

        plt.close(cm_fig)

        print(f"Fold {fold_idx} | Epoch {epoch+1} | "
              f"Train Acc: {train_accuracy:.4f} | Val Acc: {val_accuracy:.4f} | "
              f"Prec: {precision:.4f} | Rec: {recall:.4f} | ROC-AUC: {roc_auc:.4f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.6f}")

    # ------------------- FINAL EVAL WITH BEST MODEL -------------------
    if best_weights is None:
        raise RuntimeError("No best_weights found — training didn't improve over initial state.")
    model.load_state_dict(best_weights)
    print(f"✔ Loaded best model weights for fold {fold_idx} (best epoch = {best_epoch})")

    # Re-run validation on best model to get final metrics
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)

            loss = criterion(outputs, batch_y)
            val_loss += loss.item()

            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs.data, 1)

            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()

            all_labels.append(batch_y.cpu().numpy())
            all_preds.append(predicted.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    final_avg_val_loss = val_loss / len(val_loader)
    final_val_accuracy = correct / total

    all_labels = np.concatenate(all_labels)
    all_preds = np.concatenate(all_preds)
    all_probs = np.concatenate(all_probs)

    final_precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    final_recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    try:
        final_roc_auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    except:
        final_roc_auc = float("nan")

    # OPTIONAL: final confusion matrix for best model
    final_cm_fig = plot_confusion_matrix(
        all_labels,
        all_preds,
        ["Healthy Leafs", "Semilooper", "Spodoptera"],
        normalize=False
    )

    # ------------------- SAVE + UPLOAD ARTIFACT ONCE -------------------
    save_path = f"{group_name}_best_fold_{fold_idx}.pt"
    torch.save(best_weights, save_path)   # save single checkpoint once
    print(f"✔ Saved best checkpoint to {save_path}")

    if CONFIG['ACTIVE']:
        artifact = wandb.Artifact(
            name=f"{group_name}_model_fold_{fold_idx}",
            type="model",
            description=f"Best checkpoint for fold {fold_idx} (epoch={best_epoch})"
        )
        artifact.add_file(save_path)
        wandb.log_artifact(artifact)   # upload exactly once
        print("✔ Uploaded artifact to W&B")
    
        # ------------------- W&B TABLE LOG -------------------
        metrics_table = wandb.Table(
            columns=[
                "fold",
                "best_epoch",
                "best_val_loss",
                "final_val_loss",
                "final_val_accuracy",
                "final_precision",
                "final_recall",
                "final_roc_auc"
            ]
        )
    
        metrics_table.add_data(
            fold_idx,
            best_epoch,
            best_val_loss,
            final_avg_val_loss,
            final_val_accuracy,
            final_precision,
            final_recall,
            final_roc_auc
        )
    
        wandb.log({
            "best_model_metrics": metrics_table,
            "best_model_confusion_matrix": final_cm_fig
        })

        wandb.finish()
        
    plt.close(final_cm_fig)

In [ ]:
# 5. Main K-Fold Loop
suffix = "Segmented-" + str(1)
def run_k_fold():
    # Generate unique group ID
    experiment_group_name = "ResNet50-KFold-Test-" + suffix
    print(f"Starting K-Fold Experiment: {experiment_group_name}")

    # Get Data
    dataset = get_dataset()

    # K-Fold Splitter
    kf = KFold(n_splits=CONFIG["K_FOLDS"], shuffle=True, random_state=RANDOM_STATE)

    # We iterate over indices provided by KFold
    # dataset is a ConcatDataset, so we can access it via indices
    for fold_idx, (train_indices, val_indices) in enumerate(kf.split(np.arange(len(dataset)))):
        print(f"\n--- Starting Fold {fold_idx + 1}/{CONFIG['K_FOLDS']} ---")

        # Create Subsets based on indices
        train_subset = Subset(dataset, train_indices)
        val_subset = Subset(dataset, val_indices)

        # Create DataLoaders
        train_loader = DataLoader(train_subset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True)
        val_loader = DataLoader(val_subset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False)

        # Run training for this fold
        train_fold(fold_idx + 1, train_loader, val_loader, experiment_group_name)

In [ ]:
run_k_fold()